In [18]:
!pip install evaluate

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [19]:
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate

# Load the full dataset
df = pd.read_csv("/kaggle/input/nli-training/train.csv")

# Split into train/val
train_df, val_df = train_test_split(df, test_size=0.1, random_state=42, stratify=df["label"])

# Convert to HuggingFace Datasets
train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)


In [20]:

model_checkpoint = "roberta-large"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize(example):
    return tokenizer(example["premise"], example["hypothesis"], truncation=True, padding="max_length", max_length=256)

train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)


Map:   0%|          | 0/21988 [00:00<?, ? examples/s]

Map:   0%|          | 0/2444 [00:00<?, ? examples/s]

In [21]:
num_labels = 2

model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=num_labels)

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")
precision = evaluate.load("precision")
recall = evaluate.load("recall")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels, average="macro")["f1"],
        "precision": precision.compute(predictions=preds, references=labels, average="macro")["precision"],
        "recall": recall.compute(predictions=preds, references=labels, average="macro")["recall"],
    }


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [22]:
training_args = TrainingArguments(
    output_dir="./roberta-nli-fixed",
    learning_rate=2e-5,
    per_device_train_batch_size=8,          
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    
    # ✅ Logging
    logging_dir="./logs",
    logging_strategy="steps",
    logging_steps=50,

    # ✅ Evaluation + model saving
    eval_strategy="epoch",
    save_strategy="epoch",
    save_only_model=True,
    load_best_model_at_end=True,
    metric_for_best_model="f1",

    # ✅ Clean output
    report_to="none",
    disable_tqdm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()


/tmp/ipykernel_31/1551931702.py:26: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.325300,0.287812,0.897300,0.897192,0.897126,0.897273
2,0.201600,0.387080,0.910393,0.910325,0.910205,0.910556
3,0.105200,0.452089,0.914484,0.914326,0.914659,0.914105


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


TrainOutput(global_step=4125, training_loss=0.24586390350804185, metrics={'train_runtime': 6788.6974, 'train_samples_per_second': 9.717, 'train_steps_per_second': 0.608, 'total_flos': 3.0736960225284096e+16, 'train_loss': 0.24586390350804185, 'epoch': 3.0})

In [23]:
# # Clear output folder
# import os

# def remove_folder_contents(folder):
#     for the_file in os.listdir(folder):
#         file_path = os.path.join(folder, the_file)
#         try:
#             if os.path.isfile(file_path):
#                 os.unlink(file_path)
#             elif os.path.isdir(file_path):
#                 remove_folder_contents(file_path)
#                 os.rmdir(file_path)
#         except Exception as e:
#             print(e)

# folder_path = '/kaggle/working'
# remove_folder_contents(folder_path)



In [24]:
trainer.evaluate()

# Save model and tokenizer
model.save_pretrained("nli_bert_model")
tokenizer.save_pretrained("nli_bert_model")


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


('nli_bert_model/tokenizer_config.json',
 'nli_bert_model/special_tokens_map.json',
 'nli_bert_model/vocab.json',
 'nli_bert_model/merges.txt',
 'nli_bert_model/added_tokens.json',
 'nli_bert_model/tokenizer.json')

In [25]:
# Load test CSV
test_df = pd.read_csv("/kaggle/input/nli-development-dataset/dev.csv")

# Tokenize
test_ds = Dataset.from_pandas(test_df)
test_ds = test_ds.map(tokenize, batched=True)

# Predict
preds = trainer.predict(test_ds)
labels = np.argmax(preds.predictions, axis=1)

# Save predictions in required format (no header)
pd.Series(labels).to_csv("Group_52_C.csv", index=False, header=False)

Map:   0%|          | 0/6736 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


In [26]:
!zip -r nli_bert_model.zip nli_bert_model

  adding: nli_bert_model/ (stored 0%)
  adding: nli_bert_model/vocab.json (deflated 59%)
  adding: nli_bert_model/merges.txt (deflated 53%)
  adding: nli_bert_model/tokenizer.json

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


 (deflated 82%)
  adding: nli_bert_model/tokenizer_config.json (deflated 75%)
  adding: nli_bert_model/special_tokens_map.json (deflated 52%)
  adding: nli_bert_model/model.safetensors (deflated 9%)
  adding: nli_bert_model/config.json (deflated 50%)
